# Milestone 2: Naive CUDA pipeline

One kernel per stage, one GPU thread per pixel. The maths in each kernel is identical to the CPU function for the same stage. Compare them side by side: the loop over pixels disappears, because each thread handles one pixel.

The new CUDA words (`__global__`, `__device__`, `threadIdx`, `blockIdx`, `blockDim`, `dim3`, `<<< >>>`, `cudaMalloc`, `cudaMemcpy`, `cudaFree`, `cudaGetLastError`, `cudaDeviceSynchronize`) are each explained in a comment where they first appear.

**Runtime must be T4 GPU** (Runtime → Change runtime type).

## Step 1: Connect Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project
!ls

## Step 2: Write the new source files
These reuse `isp_io.h` / `isp_io.cpp` from Milestone 1, which are already in your Drive folder.
- `cuda_helpers.h` / `.cu`: `checkCudaError` and the block size.
- `gpu_kernels.h`: the list of the 5 kernels.
- `gpu_pointwise_kernels.cu`: stages 1, 3, 5. **Read the big comment at the top:** it explains how each thread finds its pixel.
- `gpu_neighbourhood_kernels.cu`: stages 2 and 4.
- `gpu_naive_main.cu`: allocates GPU memory, copies the data in, launches the kernels, copies the result back.

In [ ]:
%%writefile cuda_helpers.h
// cuda_helpers.h
// Small helpers shared by every CUDA program in this project.

#pragma once

// cuda_runtime.h declares the CUDA functions we call from the CPU side:
// cudaMalloc, cudaMemcpy, cudaFree, cudaDeviceSynchronize and so on.
#include <cuda_runtime.h>

// Every CUDA API call returns a cudaError_t. cudaSuccess means it worked.
// CUDA does NOT stop the program by itself when something fails; it just
// returns an error code that is easy to ignore. Ignored errors show up later
// as wrong pictures or confusing crashes, so we check after every call.
// If the result is an error, this prints what we were doing and exits.
void checkCudaError(cudaError_t result, const char* whatWeWereDoing);

// Block size used for every 2D kernel launch: 16 x 16 = 256 threads per block.
// 256 is a common, safe choice: big enough to keep the GPU busy, well under
// the limit of 1024 threads per block.
const int BLOCK_WIDTH = 16;
const int BLOCK_HEIGHT = 16;


In [ ]:
%%writefile cuda_helpers.cu
// cuda_helpers.cu

#include "cuda_helpers.h"

#include <cstdlib>
#include <iostream>

void checkCudaError(cudaError_t result, const char* whatWeWereDoing) {
    if (result != cudaSuccess) {
        // cudaGetErrorString turns the error code into readable text,
        // for example "out of memory" or "invalid configuration argument".
        std::cout << "CUDA ERROR while " << whatWeWereDoing << ": "
                  << cudaGetErrorString(result) << std::endl;
        std::exit(1);
    }
}


In [ ]:
%%writefile gpu_kernels.h
// gpu_kernels.h
// The five ISP stages as CUDA kernels (naive version: one thread per pixel).
// Same image layouts as the CPU version (see cpu_stages.h).
//
// "__global__" marks a KERNEL: a function that runs on the GPU but is started
// (launched) from CPU code. Thousands of copies run at the same time, one per
// thread. A kernel must return void; it writes its results into GPU memory.
//
// All pointers passed to these kernels point to GPU memory (from cudaMalloc).
// The GPU cannot read normal CPU memory such as a std::vector.

#pragma once

#include <cstdint>

// Stage 1 (gpu_pointwise_kernels.cu)
__global__ void blackLevelKernel(const uint16_t* bayerPixels, float* normalisedBayer,
                                 int imageWidth, int imageHeight,
                                 float blackLevel, float whiteLevel);

// Stage 2 (gpu_neighbourhood_kernels.cu)
__global__ void demosaicKernel(const float* normalisedBayer, float* rgbImage,
                               int imageWidth, int imageHeight);

// Stage 3 (gpu_pointwise_kernels.cu) - changes rgbImage in place.
__global__ void whiteBalanceKernel(float* rgbImage, int imageWidth, int imageHeight,
                                   float redGain, float greenGain, float blueGain);

// Stage 4 (gpu_neighbourhood_kernels.cu)
__global__ void denoiseKernel(const float* rgbImage, float* denoisedImage,
                              int imageWidth, int imageHeight);

// Stage 5 (gpu_pointwise_kernels.cu)
__global__ void gammaKernel(const float* rgbImage, unsigned char* outputBytes,
                            int imageWidth, int imageHeight);


In [ ]:
%%writefile gpu_pointwise_kernels.cu
// gpu_pointwise_kernels.cu
// Stages 1, 3 and 5 on the GPU. Compare each kernel with the CPU function of
// the same stage: the maths is identical. The only change is that the CPU
// version loops over all pixels, while here each thread handles ONE pixel
// and the loop disappears. The GPU runs thousands of threads at once instead.

#include "gpu_kernels.h"

// ---------------------------------------------------------------------------
// How a thread finds "its" pixel (used the same way in every kernel):
//
// We launch a 2D GRID of BLOCKS, and each block is a 2D group of THREADS
// (16 x 16 here). CUDA gives each thread four built-in values:
//   threadIdx.x / threadIdx.y : this thread's position INSIDE its block (0..15)
//   blockIdx.x  / blockIdx.y  : which block this thread belongs to
//   blockDim.x  / blockDim.y  : the size of a block (16 and 16)
// So the pixel column is: (block number * block width) + position in block.
// Example: block 2, thread 5, block width 16  ->  column 2*16 + 5 = 37.
//
// The grid is rounded UP to whole blocks, so a few threads at the right and
// bottom edges land outside the image. They must do nothing, which is what
// the "if (column >= imageWidth ...) return;" check is for. Without it they
// would write past the end of the buffer.
// ---------------------------------------------------------------------------

__global__ void blackLevelKernel(const uint16_t* bayerPixels, float* normalisedBayer,
                                 int imageWidth, int imageHeight,
                                 float blackLevel, float whiteLevel) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    int pixelIndex = row * imageWidth + column;
    float rawValue = static_cast<float>(bayerPixels[pixelIndex]);
    float usefulRange = whiteLevel - blackLevel;
    float normalisedValue = (rawValue - blackLevel) / usefulRange;

    if (normalisedValue < 0.0f) {
        normalisedValue = 0.0f;
    }
    if (normalisedValue > 1.0f) {
        normalisedValue = 1.0f;
    }

    normalisedBayer[pixelIndex] = normalisedValue;
}

__global__ void whiteBalanceKernel(float* rgbImage, int imageWidth, int imageHeight,
                                   float redGain, float greenGain, float blueGain) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    int pixelIndex = row * imageWidth + column;
    int redIndex = pixelIndex * 3 + 0;
    int greenIndex = pixelIndex * 3 + 1;
    int blueIndex = pixelIndex * 3 + 2;

    float redValue = rgbImage[redIndex] * redGain;
    float greenValue = rgbImage[greenIndex] * greenGain;
    float blueValue = rgbImage[blueIndex] * blueGain;

    if (redValue > 1.0f) {
        redValue = 1.0f;
    }
    if (greenValue > 1.0f) {
        greenValue = 1.0f;
    }
    if (blueValue > 1.0f) {
        blueValue = 1.0f;
    }

    // Writing back in place is safe here: each thread touches only its own
    // pixel, so no thread ever reads a value another thread has changed.
    rgbImage[redIndex] = redValue;
    rgbImage[greenIndex] = greenValue;
    rgbImage[blueIndex] = blueValue;
}

// "__device__" marks a helper function that runs on the GPU and can only be
// called from GPU code (from a kernel or another __device__ function).
// Unlike a __global__ kernel, it can return a value.
__device__ unsigned char gammaToByte(float linearValue) {
    if (linearValue < 0.0f) {
        linearValue = 0.0f;
    }
    if (linearValue > 1.0f) {
        linearValue = 1.0f;
    }

    // powf is the float version of pow on the GPU. Its result can differ from
    // the CPU's std::pow in the last tiny digit; Milestone 3 looks at this.
    float gammaValue = powf(linearValue, 1.0f / 2.2f);
    float scaledValue = gammaValue * 255.0f + 0.5f;
    return static_cast<unsigned char>(scaledValue);
}

__global__ void gammaKernel(const float* rgbImage, unsigned char* outputBytes,
                            int imageWidth, int imageHeight) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    // One thread per pixel, so each thread converts all three channels.
    int pixelIndex = row * imageWidth + column;
    for (int channel = 0; channel < 3; channel++) {
        int valueIndex = pixelIndex * 3 + channel;
        outputBytes[valueIndex] = gammaToByte(rgbImage[valueIndex]);
    }
}


In [ ]:
%%writefile gpu_neighbourhood_kernels.cu
// gpu_neighbourhood_kernels.cu
// Stages 2 and 4 on the GPU. Same maths and the same border rules as
// cpu_neighbourhood_stages.cpp (see the comments there for WHY reflect/clamp).
// Naive version: every thread reads its 3x3 neighbours straight from global
// memory (the GPU's large, slow main memory). Neighbouring threads therefore
// read many of the same values again and again. Milestone 6 fixes that.

#include "gpu_kernels.h"

// Same as reflectIndex on the CPU, marked __device__ so GPU code can call it.
__device__ int reflectIndexGpu(int index, int size) {
    if (index < 0) {
        return -index;
    }
    if (index >= size) {
        return 2 * size - 2 - index;
    }
    return index;
}

__device__ int clampIndexGpu(int index, int size) {
    if (index < 0) {
        return 0;
    }
    if (index >= size) {
        return size - 1;
    }
    return index;
}

__device__ float readBayerValueGpu(const float* bayerImage, int imageWidth, int imageHeight,
                                   int row, int column) {
    int safeRow = reflectIndexGpu(row, imageHeight);
    int safeColumn = reflectIndexGpu(column, imageWidth);
    int pixelIndex = safeRow * imageWidth + safeColumn;
    return bayerImage[pixelIndex];
}

__global__ void demosaicKernel(const float* normalisedBayer, float* rgbImage,
                               int imageWidth, int imageHeight) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    float centre = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row, column);
    float up = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row - 1, column);
    float down = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row + 1, column);
    float left = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row, column - 1);
    float right = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row, column + 1);
    float upLeft = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row - 1, column - 1);
    float upRight = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row - 1, column + 1);
    float downLeft = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row + 1, column - 1);
    float downRight = readBayerValueGpu(normalisedBayer, imageWidth, imageHeight, row + 1, column + 1);

    float crossAverage = (up + down + left + right) / 4.0f;
    float diagonalAverage = (upLeft + upRight + downLeft + downRight) / 4.0f;
    float horizontalAverage = (left + right) / 2.0f;
    float verticalAverage = (up + down) / 2.0f;

    bool isEvenRow = (row % 2 == 0);
    bool isEvenColumn = (column % 2 == 0);

    float redValue = 0.0f;
    float greenValue = 0.0f;
    float blueValue = 0.0f;

    if (isEvenRow && isEvenColumn) {
        redValue = centre;
        greenValue = crossAverage;
        blueValue = diagonalAverage;
    } else if (isEvenRow && !isEvenColumn) {
        redValue = horizontalAverage;
        greenValue = centre;
        blueValue = verticalAverage;
    } else if (!isEvenRow && isEvenColumn) {
        redValue = verticalAverage;
        greenValue = centre;
        blueValue = horizontalAverage;
    } else {
        redValue = diagonalAverage;
        greenValue = crossAverage;
        blueValue = centre;
    }

    int pixelIndex = row * imageWidth + column;
    rgbImage[pixelIndex * 3 + 0] = redValue;
    rgbImage[pixelIndex * 3 + 1] = greenValue;
    rgbImage[pixelIndex * 3 + 2] = blueValue;
}

__global__ void denoiseKernel(const float* rgbImage, float* denoisedImage,
                              int imageWidth, int imageHeight) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    const float gaussianWeights[3][3] = {
        {1.0f, 2.0f, 1.0f},
        {2.0f, 4.0f, 2.0f},
        {1.0f, 2.0f, 1.0f}
    };
    const float weightSum = 16.0f;

    int pixelIndex = row * imageWidth + column;

    for (int channel = 0; channel < 3; channel++) {
        float weightedTotal = 0.0f;

        for (int rowOffset = -1; rowOffset <= 1; rowOffset++) {
            for (int columnOffset = -1; columnOffset <= 1; columnOffset++) {
                int neighbourRow = clampIndexGpu(row + rowOffset, imageHeight);
                int neighbourColumn = clampIndexGpu(column + columnOffset, imageWidth);
                int neighbourPixel = neighbourRow * imageWidth + neighbourColumn;
                float neighbourValue = rgbImage[neighbourPixel * 3 + channel];
                float weight = gaussianWeights[rowOffset + 1][columnOffset + 1];
                weightedTotal = weightedTotal + neighbourValue * weight;
            }
        }

        denoisedImage[pixelIndex * 3 + channel] = weightedTotal / weightSum;
    }
}


In [ ]:
%%writefile gpu_naive_main.cu
// gpu_naive_main.cu
// Milestone 2: the naive GPU pipeline.
// Plan: copy the Bayer data to the GPU ONCE, run all 5 kernels one after
// another with every intermediate image staying in GPU memory, then copy the
// final 8-bit image back ONCE. Copies between CPU and GPU are slow, so we
// avoid copying anything in between stages.

#include <iostream>
#include <vector>

#include "cuda_helpers.h"
#include "gpu_kernels.h"
#include "isp_io.h"

int main() {
    // ---- Load input on the CPU (same files as the CPU version) ----
    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;
    int pixelCount = imageWidth * imageHeight;
    std::cout << "Image: " << imageWidth << " x " << imageHeight << std::endl;

    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }

    // ---- Work out the size of each buffer in bytes ----
    // size_t is an unsigned integer type big enough for any memory size.
    size_t bayerBytes = static_cast<size_t>(pixelCount) * sizeof(uint16_t);
    size_t normalisedBytes = static_cast<size_t>(pixelCount) * sizeof(float);
    size_t rgbFloatBytes = static_cast<size_t>(pixelCount) * 3 * sizeof(float);
    size_t outputBytesCount = static_cast<size_t>(pixelCount) * 3 * sizeof(unsigned char);

    // ---- Allocate GPU memory ----
    // cudaMalloc reserves memory on the GPU (like malloc, but on the GPU) and
    // stores its address in our pointer. We pass the ADDRESS of the pointer
    // (&deviceBayer) so cudaMalloc can fill it in. We name GPU pointers
    // "device..." because CPU code must never read through them directly.
    uint16_t* deviceBayer = nullptr;
    float* deviceNormalised = nullptr;
    float* deviceRgb = nullptr;
    float* deviceDenoised = nullptr;
    unsigned char* deviceOutput = nullptr;

    checkCudaError(cudaMalloc(&deviceBayer, bayerBytes), "allocating Bayer buffer");
    checkCudaError(cudaMalloc(&deviceNormalised, normalisedBytes), "allocating normalised buffer");
    checkCudaError(cudaMalloc(&deviceRgb, rgbFloatBytes), "allocating RGB buffer");
    checkCudaError(cudaMalloc(&deviceDenoised, rgbFloatBytes), "allocating denoised buffer");
    checkCudaError(cudaMalloc(&deviceOutput, outputBytesCount), "allocating output buffer");

    // ---- Copy the input to the GPU (once) ----
    // cudaMemcpy(destination, source, byte count, direction).
    // cudaMemcpyHostToDevice means CPU memory ("host") to GPU memory ("device").
    checkCudaError(cudaMemcpy(deviceBayer, bayerPixels.data(), bayerBytes, cudaMemcpyHostToDevice),
                   "copying Bayer data to the GPU");

    // ---- Choose the launch shape ----
    // dim3 is a CUDA type holding up to 3 sizes (x, y, z); unset ones are 1.
    // blockShape: 16 x 16 threads per block.
    // gridShape: enough blocks to cover the image. The "+ BLOCK_WIDTH - 1"
    // rounds UP, so a partial block covers the leftover pixels at the edge.
    // Example: width 5796 / 16 = 362.25, so we need 363 blocks across.
    dim3 blockShape(BLOCK_WIDTH, BLOCK_HEIGHT);
    int blocksAcross = (imageWidth + BLOCK_WIDTH - 1) / BLOCK_WIDTH;
    int blocksDown = (imageHeight + BLOCK_HEIGHT - 1) / BLOCK_HEIGHT;
    dim3 gridShape(blocksAcross, blocksDown);
    std::cout << "Grid: " << blocksAcross << " x " << blocksDown << " blocks of "
              << BLOCK_WIDTH << " x " << BLOCK_HEIGHT << " threads" << std::endl;

    // ---- Run the 5 kernels ----
    // kernelName<<<gridShape, blockShape>>>(arguments) LAUNCHES a kernel: it
    // starts gridShape x blockShape threads, all running the kernel at once.
    // The launch returns to the CPU immediately, without waiting for the GPU.
    // Kernels launched one after another still run IN ORDER on the GPU, so
    // stage 2 never starts before stage 1 has finished.
    //
    // A launch itself returns nothing, so we ask cudaGetLastError() whether
    // the launch was accepted (for example, a bad block size is caught here).
    blackLevelKernel<<<gridShape, blockShape>>>(deviceBayer, deviceNormalised,
                                                imageWidth, imageHeight,
                                                metadata.blackLevel, metadata.whiteLevel);
    checkCudaError(cudaGetLastError(), "launching blackLevelKernel");

    demosaicKernel<<<gridShape, blockShape>>>(deviceNormalised, deviceRgb, imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching demosaicKernel");

    whiteBalanceKernel<<<gridShape, blockShape>>>(deviceRgb, imageWidth, imageHeight,
                                                  metadata.whiteBalanceRed,
                                                  metadata.whiteBalanceGreen,
                                                  metadata.whiteBalanceBlue);
    checkCudaError(cudaGetLastError(), "launching whiteBalanceKernel");

    denoiseKernel<<<gridShape, blockShape>>>(deviceRgb, deviceDenoised, imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching denoiseKernel");

    gammaKernel<<<gridShape, blockShape>>>(deviceDenoised, deviceOutput, imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching gammaKernel");

    // cudaDeviceSynchronize makes the CPU wait until the GPU has finished ALL
    // the work launched so far. Errors that happen WHILE a kernel runs (for
    // example reading outside a buffer) are only reported here.
    checkCudaError(cudaDeviceSynchronize(), "running the kernels");

    // ---- Copy the result back to the CPU (once) ----
    // cudaMemcpyDeviceToHost is the opposite direction: GPU to CPU.
    std::vector<unsigned char> outputBytes(outputBytesCount);
    checkCudaError(cudaMemcpy(outputBytes.data(), deviceOutput, outputBytesCount, cudaMemcpyDeviceToHost),
                   "copying the output back to the CPU");

    // ---- Free GPU memory ----
    // cudaFree gives GPU memory back, like free() does on the CPU.
    checkCudaError(cudaFree(deviceBayer), "freeing Bayer buffer");
    checkCudaError(cudaFree(deviceNormalised), "freeing normalised buffer");
    checkCudaError(cudaFree(deviceRgb), "freeing RGB buffer");
    checkCudaError(cudaFree(deviceDenoised), "freeing denoised buffer");
    checkCudaError(cudaFree(deviceOutput), "freeing output buffer");

    bool writeOk = writeRgbFile("output/gpu_naive_output.bin", outputBytes);
    if (!writeOk) {
        return 1;
    }
    std::cout << "Saved output/gpu_naive_output.bin" << std::endl;
    return 0;
}


## Step 3: Compile with nvcc
`nvcc` is NVIDIA's compiler. It splits each `.cu` file into CPU code (passed to the normal C++ compiler) and GPU code (compiled for the GPU). `-arch=sm_75` says "build GPU code for compute capability 7.5", which is the T4. `isp_io.cpp` is plain C++, and nvcc handles it too.

**Expected:** no output. Silence means success.

In [ ]:
!nvcc -O2 -arch=sm_75 -o /content/gpu_naive gpu_naive_main.cu gpu_pointwise_kernels.cu gpu_neighbourhood_kernels.cu cuda_helpers.cu isp_io.cpp

## Step 4: Run (CHECKPOINT 1)
**Expected:**
```
Image: 5796 x 3870
Grid: 363 x 242 blocks of 16 x 16 threads
Saved output/gpu_naive_output.bin
```
(5796 / 16 = 362.25, rounded up to 363. 3870 / 16 = 241.9, rounded up to 242.) That's 87,846 blocks of 256 threads, about 22.5 million threads, one per pixel plus a few spares at the edges.

There's no timing yet. Milestone 4 measures the GPU properly, because timing GPU code with a normal clock is misleading.

In [ ]:
!/content/gpu_naive

## Step 5: View the GPU result next to the CPU result (CHECKPOINT 2)
**Expected:** the two photos look identical. Milestone 3 checks this number by number.

In [ ]:
!python view_output.py output/gpu_naive_output.bin output/gpu_naive_output.png
from PIL import Image
from IPython.display import display
for imagePath in ['output/cpu_output.png', 'output/gpu_naive_output.png']:
    print(imagePath)
    previewImage = Image.open(imagePath)
    previewImage.thumbnail((700, 700))
    display(previewImage)